### Caso não tenha as libs instaladas no Kernel

In [1]:
%pip install --force-reinstall "numpy>=1.26,<2.0" "scipy>=1.11,<1.14" "scikit-learn>=1.4,<1.6" "opencv-python>=4.9,<5.0" "pandas>=2.2,<3.0" "plotly>=6.1,<7.0"
%pip install --upgrade nbformat


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Import das libs

In [7]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import math
import warnings
import glob
import os
from pathlib import Path

import cv2
from IPython.display import Markdown, display
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


### Análise dos dados

In [3]:
lista_dfs = []
caminhos_ficheiros = glob.glob('../logs/*.csv') or glob.glob('logs/*.csv')

for caminho in caminhos_ficheiros:
    df_temp = pd.read_csv(caminho)
    
    df_temp['altitude'] = -df_temp['z']
    
    lista_dfs.append(df_temp)

#### Análise da trajetória

In [4]:
for caminho, df in zip(caminhos_ficheiros, lista_dfs):
    
    # Extraindo o timestamp do nome do arquivo
    nome_arquivo = os.path.basename(caminho) 
    timestamp = nome_arquivo.replace('voo_teste_', '').replace('.csv', '')
    
    fig_3d = go.Figure()

    # Adiciona a linha da Trajetória Real
    fig_3d.add_trace(go.Scatter3d(
        x=df['x'], y=df['y'], z=df['altitude'],
        mode='lines',
        line=dict(color='royalblue', width=4),
        name='Trajetória Real (Odometria)'
    ))

    # Adiciona o Ponto de Partida
    fig_3d.add_trace(go.Scatter3d(
        x=[df['x'].iloc[0]], y=[df['y'].iloc[0]], z=[df['altitude'].iloc[0]],
        mode='markers',
        marker=dict(color='green', size=6),
        name='Ponto de Partida'
    ))

    # Adiciona o Ponto Final
    fig_3d.add_trace(go.Scatter3d(
        x=[df['x'].iloc[-1]], y=[df['y'].iloc[-1]], z=[df['altitude'].iloc[-1]],
        mode='markers',
        marker=dict(color='red', size=6, symbol='x'),
        name='Ponto Final da Run'
    ))

    fig_3d.update_layout(
        title=f'Análise de Trajetória 3D do VANT - Run: {timestamp}',
        scene=dict(
            xaxis_title='Posição X (Metros)',
            yaxis_title='Posição Y (Metros)',
            zaxis_title='Posição Z / Altitude (Metros)',
            camera=dict(eye=dict(x=1.5, y=1.5, z=0.5)) 
        ),
        legend=dict(x=0, y=1),
        margin=dict(l=0, r=0, b=0, t=40) 
    )

    fig_3d.show()

#### Análise do Pitch and Roll

In [5]:
for caminho, df in zip(caminhos_ficheiros, lista_dfs):
    
    # Extraindo o timestamp do nome do arquivo
    nome_arquivo = os.path.basename(caminho) 
    timestamp = nome_arquivo.replace('voo_teste_', '').replace('.csv', '')
    
    fig_2d = go.Figure()

    # Adiciona a linha de Roll
    fig_2d.add_trace(go.Scatter(
        x=df['timestamp'], y=df['roll_speed'],
        mode='lines',
        name='Roll',
        opacity=0.7
    ))

    # Adiciona a linha de Pitch
    fig_2d.add_trace(go.Scatter(
        x=df['timestamp'], y=df['pitch_speed'],
        mode='lines',
        name='Pitch',
        opacity=0.7
    ))

    fig_2d.update_layout(
        title=f'Esforço de Controle: Velocidades Angulares - Run: {timestamp}',
        xaxis_title='Tempo de Voo (Segundos)',
        yaxis_title='Velocidade Angular (rad/s)',
        template='plotly_white',
        hovermode='x unified' # Cria uma linha vertical interativa ao passar o mouse
    )

    fig_2d.show()

#### Analise do depth ground truth do Gazebo

Esta analise usa os pares RGB/depth salvos em `datasets/depth_ground_truth/run_*/metadata.csv` para transformar o depth renderizado pelo Gazebo em metricas por frame. A ideia e medir quando a cena ficou visualmente proxima da camera monocular e cruzar isso com IMU/atitude, deixando pronto o terreno para o treino futuro do modelo monocular.

Fontes Documentação: 
[Gazebo DepthCamera](https://gazebosim.org/api/rendering/7/classgz_1_1rendering_1_1DepthCamera.html), 
[NumPy percentile](https://numpy.org/doc/stable/reference/generated/numpy.percentile.html), 
[Plotly multiple axes](https://plotly.com/python/multiple-axes/), 

Fontes Artigos: 
[RealTimeMonocular2022](https://doi.org/10.1109/TITS.2022.3160741), 
[Vyas2022](https://doi.org/10.48550/arXiv.2205.01399), 
[Tarrio2015](https://doi.org/10.1109/iccv.2015.87).

In [6]:
def localizar_raiz_projeto_depth():
    '''
    Localiza a raiz do projeto a partir do notebook aberto.

    O notebook pode ser executado a partir de estudos_e_analises/ ou da raiz do
    repositorio. A funcao procura a pasta datasets/depth_ground_truth nesses niveis
    para evitar caminhos absolutos presos ao Windows ou ao WSL.

    Fontes:
    [Python pathlib] https://docs.python.org/3/library/pathlib.html
    [Gazebo DepthCamera] https://gazebosim.org/api/rendering/7/classgz_1_1rendering_1_1DepthCamera.html
    '''

    candidatos = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidato in candidatos:
        if (candidato / 'datasets' / 'depth_ground_truth').exists():
            return candidato
    return Path.cwd()


def localizar_ultima_run_depth(base_dir):
    '''
    Retorna o metadata.csv da run de depth ground truth mais recente.

    A pasta gerada pelo controlador segue o padrao run_<data_hora>. Usar a ultima run
    facilita reexecutar a analise logo depois de um voo sem editar o notebook.

    Fontes:
    [Python pathlib glob] https://docs.python.org/3/library/pathlib.html#pathlib.Path.glob
    [Pandas read_csv] https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
    '''

    depth_dir = base_dir / 'datasets' / 'depth_ground_truth'
    metadados = sorted(depth_dir.glob('run_*/metadata.csv'))
    return metadados[-1] if metadados else None


def resolver_arquivo_depth(depth_path, run_dir):
    '''
    Resolve o caminho do arquivo NPY mesmo quando o CSV veio do WSL.

    O metadata.csv pode armazenar caminhos como /home/prograf4080/...; quando eles nao
    existem no ambiente atual, a funcao reconstrui o caminho local usando run_dir/depth_m
    e o nome do arquivo.

    Fontes:
    [Python pathlib] https://docs.python.org/3/library/pathlib.html
    [NumPy load] https://numpy.org/doc/stable/reference/generated/numpy.load.html
    '''

    caminho_original = Path(str(depth_path))
    if caminho_original.exists():
        return caminho_original
    return run_dir / 'depth_m' / caminho_original.name


def enriquecer_metadata_depth(df, run_dir, limiares=(2.0, 5.0, 10.0)):
    '''
    Calcula percentis e porcentagens de pixels proximos para cada mapa de depth.

    A funcao le os arquivos NPY em metros, descarta valores invalidos e mede P10/P50/P90
    alem da fracao de pixels abaixo de limiares de proximidade. Essas metricas convertem
    o depth do simulador em sinais mais simples para comparar com IMU, pan compensado e
    futuro treinamento monocular.

    Fontes:
    [NumPy percentile] https://numpy.org/doc/stable/reference/generated/numpy.percentile.html
    [Artigo - RealTimeMonocular2022] https://doi.org/10.1109/TITS.2022.3160741
    [Artigo - Vyas2022] https://doi.org/10.48550/arXiv.2205.01399
    '''

    df = df.copy()
    if 'sample_id' in df.columns:
        df['sample_id'] = df['sample_id'].astype(str).str.zfill(6)

    metricas = {
        'depth_p10_m': [],
        'depth_p50_m': [],
        'depth_p90_m': [],
        'valid_px_pct': [],
    }
    for limiar in limiares:
        metricas[f'depth_close_{int(limiar)}m_pct'] = []

    for _, row in df.iterrows():
        arquivo_depth = resolver_arquivo_depth(row.get('depth_path', ''), run_dir)
        if not arquivo_depth.exists():
            for valores in metricas.values():
                valores.append(np.nan)
            continue

        depth = np.load(arquivo_depth, mmap_mode='r')
        validos = np.isfinite(depth) & (depth > 0.0)
        valores_depth = np.asarray(depth[validos], dtype=float)

        if valores_depth.size == 0:
            metricas['depth_p10_m'].append(np.nan)
            metricas['depth_p50_m'].append(np.nan)
            metricas['depth_p90_m'].append(np.nan)
            metricas['valid_px_pct'].append(0.0)
            for limiar in limiares:
                metricas[f'depth_close_{int(limiar)}m_pct'].append(np.nan)
            continue

        metricas['depth_p10_m'].append(float(np.percentile(valores_depth, 10)))
        metricas['depth_p50_m'].append(float(np.percentile(valores_depth, 50)))
        metricas['depth_p90_m'].append(float(np.percentile(valores_depth, 90)))
        metricas['valid_px_pct'].append(float(validos.mean() * 100.0))
        for limiar in limiares:
            metricas[f'depth_close_{int(limiar)}m_pct'].append(float((valores_depth < limiar).mean() * 100.0))

    for coluna, valores in metricas.items():
        df[coluna] = valores

    return df


raiz_projeto = localizar_raiz_projeto_depth()
metadata_depth = localizar_ultima_run_depth(raiz_projeto)

if metadata_depth is None:
    display(Markdown('Nenhuma run de depth encontrada em `datasets/depth_ground_truth`.'))
else:
    depth_df = pd.read_csv(metadata_depth)
    colunas_numericas = [
        'rgb_timestamp_s', 'depth_timestamp_s', 'depth_age_s',
        'x', 'y', 'z', 'roll', 'pitch', 'yaw',
        'gyro_x', 'gyro_y', 'gyro_z', 'accel_x', 'accel_y', 'accel_z',
        'depth_min_m', 'depth_mean_m', 'depth_max_m', 'pan_comp_delta_rad'
    ]
    for coluna in colunas_numericas:
        if coluna in depth_df.columns:
            depth_df[coluna] = pd.to_numeric(depth_df[coluna], errors='coerce')

    if 'pan_comp_delta_rad' not in depth_df.columns:
        depth_df['pan_comp_delta_rad'] = np.nan

    depth_df = depth_df.sort_values('rgb_timestamp_s').reset_index(drop=True)
    depth_df['tempo_s'] = depth_df['rgb_timestamp_s'] - depth_df['rgb_timestamp_s'].iloc[0]
    depth_df['gyro_norm'] = np.sqrt(depth_df['gyro_x']**2 + depth_df['gyro_y']**2 + depth_df['gyro_z']**2)
    depth_df['accel_norm'] = np.sqrt(depth_df['accel_x']**2 + depth_df['accel_y']**2 + depth_df['accel_z']**2)
    depth_df = enriquecer_metadata_depth(depth_df, metadata_depth.parent)

    print(f'Run analisada: {metadata_depth.parent.name}')
    print(f'Amostras RGB/depth: {len(depth_df)}')
    print(f'Desalinhamento medio RGB-depth: {depth_df["depth_age_s"].mean() * 1000:.1f} ms')

    resumo_depth = [
        'depth_age_s', 'depth_min_m', 'depth_mean_m', 'depth_p10_m', 'depth_p50_m',
        'depth_close_2m_pct', 'depth_close_5m_pct', 'depth_close_10m_pct',
        'gyro_norm', 'accel_norm', 'pan_comp_delta_rad'
    ]
    display(depth_df[[c for c in resumo_depth if c in depth_df.columns]].describe().T)

    fig_depth = go.Figure()
    fig_depth.add_trace(go.Scatter(x=depth_df['tempo_s'], y=depth_df['depth_mean_m'], mode='lines', name='depth media'))
    fig_depth.add_trace(go.Scatter(x=depth_df['tempo_s'], y=depth_df['depth_p10_m'], mode='lines', name='depth P10'))
    fig_depth.add_trace(go.Scatter(x=depth_df['tempo_s'], y=depth_df['depth_p50_m'], mode='lines', name='depth P50'))
    fig_depth.add_trace(go.Scatter(x=depth_df['tempo_s'], y=depth_df['depth_close_5m_pct'], mode='lines', name='pixels < 5 m (%)', yaxis='y2'))
    fig_depth.update_layout(
        title='Depth ground truth: distancia da cena e area proxima',
        xaxis_title='Tempo desde o primeiro par (s)',
        yaxis=dict(title='Profundidade (m)'),
        yaxis2=dict(title='Pixels < 5 m (%)', overlaying='y', side='right'),
        template='plotly_white',
        hovermode='x unified'
    )
    fig_depth.show()

    cor_pontos = depth_df['pan_comp_delta_rad'].abs()
    titulo_cor = '|pan compensado| (rad)'
    if cor_pontos.isna().all():
        cor_pontos = depth_df['depth_mean_m']
        titulo_cor = 'depth media (m)'

    fig_imu = go.Figure()
    fig_imu.add_trace(go.Scatter(
        x=depth_df['depth_close_5m_pct'],
        y=depth_df['gyro_norm'],
        mode='markers',
        marker=dict(size=8, color=cor_pontos, colorscale='Turbo', colorbar=dict(title=titulo_cor)),
        text=depth_df['sample_id'],
        customdata=np.stack([depth_df['tempo_s'], depth_df['depth_p10_m'], depth_df['accel_norm']], axis=-1),
        hovertemplate='amostra=%{text}<br>t=%{customdata[0]:.2f}s<br>pixels < 5m=%{x:.2f}%<br>gyro=%{y:.3f}rad/s<br>depth P10=%{customdata[1]:.2f}m<br>accel=%{customdata[2]:.2f}m/s2<extra></extra>'
    ))
    fig_imu.update_layout(
        title='Frames criticos: proximidade visual x giro do drone',
        xaxis_title='Pixels validos com depth < 5 m (%)',
        yaxis_title='Norma do giroscopio (rad/s)',
        template='plotly_white'
    )
    fig_imu.show()

    colunas_ranking = [
        'sample_id', 'tempo_s', 'depth_mean_m', 'depth_p10_m', 'depth_close_2m_pct',
        'depth_close_5m_pct', 'gyro_norm', 'accel_norm', 'pan_comp_delta_rad'
    ]
    ranking = depth_df.sort_values(['depth_close_5m_pct', 'gyro_norm'], ascending=False)
    display(Markdown('**Frames mais interessantes para inspecao/treino:**'))
    display(ranking[[c for c in colunas_ranking if c in ranking.columns]].head(12))

Run analisada: run_20260516_201713
Amostras RGB/depth: 68
Desalinhamento medio RGB-depth: 47.6 ms


,count,mean,std,min,25%,50%,75%,max
depth_age_s,68.0,0.047588,0.016646,0.032000,0.032000,0.036000,0.064000,0.068000
depth_min_m,68.0,0.388854,0.526530,0.100001,0.100003,0.129423,0.164691,1.904671
depth_mean_m,68.0,16.976876,5.597741,2.605905,13.919374,16.014973,20.091402,31.782751
depth_p10_m,68.0,1.578014,0.806854,0.242897,0.750754,1.837454,2.129993,2.926932
depth_p50_m,68.0,4.791455,2.908541,0.491241,1.762810,5.132415,6.468093,11.196620
depth_close_2m_pct,68.0,26.525409,26.493500,0.571669,6.995741,12.177797,53.030628,83.561796
depth_close_5m_pct,68.0,52.168760,17.909995,17.393619,40.924597,48.677550,67.693573,95.068307
depth_close_10m_pct,68.0,70.026867,10.212742,43.529356,64.162938,70.567812,75.408443,96.298264
gyro_norm,68.0,0.848451,0.666727,0.000632,0.265517,0.801115,1.349640,2.616043
accel_norm,68.0,10.565954,0.863726,7.029227,9.955285,10.559024,11.088078,12.318418


**Frames mais interessantes para inspecao/treino:**

,sample_id,tempo_s,depth_mean_m,depth_p10_m,depth_close_2m_pct,depth_close_5m_pct,gyro_norm,accel_norm,pan_comp_delta_rad
9,000010,5.480,2.605905,0.242897,83.561796,95.068307,0.240919,10.961549,NaN
10,000011,5.908,10.251402,0.602778,69.972863,83.616925,1.341200,10.393215,NaN
8,000009,5.116,11.697951,0.298283,76.081370,81.710941,1.167928,10.180284,NaN
7,000008,4.620,13.757941,0.271112,74.334356,78.491293,0.134580,9.773060,NaN
1,000002,0.496,14.128482,0.281302,73.485017,77.910971,0.000665,9.795398,NaN
0,000001,0.000,14.128621,0.281331,73.484509,77.910156,0.001004,9.783504,NaN
2,000003,1.024,14.128729,0.281324,73.484586,77.909928,0.000720,9.789792,NaN
3,000004,1.816,14.122168,0.281341,73.485805,77.906987,0.000833,9.791848,NaN
4,000005,2.212,14.518847,0.285846,72.903302,77.378924,0.001320,9.803117,NaN
6,000007,3.168,14.559790,0.286545,72.826164,77.313489,0.000632,9.788296,NaN


#### Treino baseline com MLP simples para depth/risco

Esta celula implementa o baseline ate a etapa 5 do plano: monta um dataset supervisionado com RGB monocular, optical flow resumido e IMU; calcula alvos a partir do depth ground truth do Gazebo; separa treino/validacao/teste; treina uma MLP simples; e compara o erro contra um baseline de media. O objetivo aqui nao e prever um mapa denso perfeito, mas testar se uma MLP pequena consegue inferir indicadores uteis de profundidade/risco.

Fontes: [scikit-learn MLPRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html), [scikit-learn Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), [OpenCV Farneback Optical Flow](https://docs.opencv.org/4.x/d4/dee/tutorial_optical_flow.html), [OpenCV Canny](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html), [RealTimeMonocular2022](https://doi.org/10.1109/TITS.2022.3160741), [Vyas2022](https://doi.org/10.48550/arXiv.2205.01399), [Tarrio2015](https://doi.org/10.1109/iccv.2015.87).


In [8]:
import math
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def localizar_raiz_projeto_mlp():
    '''
    Localiza a raiz do repositorio para o treino MLP.

    O notebook pode ser executado a partir da raiz do projeto ou da pasta
    estudos_e_analises. A funcao procura datasets/depth_ground_truth nos ancestrais mais
    proximos para evitar caminhos absolutos presos ao Windows ou ao WSL.

    Fontes:
    [Python pathlib] https://docs.python.org/3/library/pathlib.html
    [Gazebo DepthCamera] https://gazebosim.org/api/rendering/7/classgz_1_1rendering_1_1DepthCamera.html
    '''

    candidatos = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidato in candidatos:
        if (candidato / 'datasets' / 'depth_ground_truth').exists():
            return candidato
    return Path.cwd()


def listar_metadados_depth_mlp(base_dir):
    '''
    Lista as runs de depth ground truth disponiveis para treinamento.

    Cada metadata.csv define uma run supervisionada. O identificador da pasta run_* e usado
    como grupo de validacao para evitar misturar frames quase identicos entre treino e teste
    quando houver varias coletas.

    Fontes:
    [Python pathlib glob] https://docs.python.org/3/library/pathlib.html#pathlib.Path.glob
    [Pandas read_csv] https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
    '''

    return sorted((base_dir / 'datasets' / 'depth_ground_truth').glob('run_*/metadata.csv'))


def resolver_arquivo_da_run(valor_caminho, run_dir, subdir):
    '''
    Resolve caminhos absolutos do CSV ou reconstrui o caminho dentro da run local.

    Os metadados podem ter sido gravados no WSL com caminhos /home/prograf4080/..., mas a
    analise pode estar rodando no Windows. Quando o caminho original nao existe, a funcao
    usa apenas o nome do arquivo dentro de rgb/ ou depth_m/.

    Fontes:
    [Python pathlib] https://docs.python.org/3/library/pathlib.html
    [NumPy load] https://numpy.org/doc/stable/reference/generated/numpy.load.html
    '''

    caminho = Path(str(valor_caminho))
    if caminho.exists():
        return caminho
    return run_dir / subdir / caminho.name


def carregar_rgb_mlp(row, run_dir):
    '''
    Carrega a imagem RGB/BGR associada a uma linha do metadata.csv.

    O OpenCV entrega a imagem em BGR. Essa representacao e mantida porque as features usam
    conversoes internas do proprio OpenCV para cinza e HSV.

    Fontes:
    [OpenCV imread] https://docs.opencv.org/4.x/d4/da8/group__imgcodecs.html
    [OpenCV color conversions] https://docs.opencv.org/4.x/de/d25/imgproc_color_conversions.html
    '''

    caminho_rgb = resolver_arquivo_da_run(row.get('rgb_path', ''), run_dir, 'rgb')
    imagem = cv2.imread(str(caminho_rgb), cv2.IMREAD_COLOR)
    if imagem is None:
        raise FileNotFoundError(f'RGB nao encontrado ou invalido: {caminho_rgb}')
    return imagem


def calcular_alvos_depth_mlp(row, run_dir, limiares=(2.0, 5.0, 10.0)):
    '''
    Calcula os alvos supervisionados da MLP a partir do mapa de profundidade.

    Os alvos sao percentis de profundidade e porcentagens de pixels mais proximos que alguns
    limiares. Essa escolha transforma o mapa denso do Gazebo em sinais compactos, adequados
    para uma MLP simples.

    Fontes:
    [NumPy percentile] https://numpy.org/doc/stable/reference/generated/numpy.percentile.html
    [Artigo - RealTimeMonocular2022] https://doi.org/10.1109/TITS.2022.3160741
    [Artigo - Vyas2022] https://doi.org/10.48550/arXiv.2205.01399
    '''

    caminho_depth = resolver_arquivo_da_run(row.get('depth_path', ''), run_dir, 'depth_m')
    depth = np.load(caminho_depth, mmap_mode='r')
    validos = np.isfinite(depth) & (depth > 0.0)
    valores = np.asarray(depth[validos], dtype=float)
    if valores.size == 0:
        raise ValueError(f'Depth sem pixels validos: {caminho_depth}')

    alvos = {
        'depth_p10_m': float(np.percentile(valores, 10)),
        'depth_p50_m': float(np.percentile(valores, 50)),
        'depth_p90_m': float(np.percentile(valores, 90)),
    }
    for limiar in limiares:
        alvos[f'depth_close_{int(limiar)}m_pct'] = float((valores < limiar).mean() * 100.0)
    return alvos


def extrair_features_visuais_mlp(imagem_bgr, gray_anterior=None, grid=(4, 3), tamanho=(96, 72)):
    '''
    Extrai features visuais compactas da imagem monocular e do optical flow.

    A imagem e reduzida para uma resolucao pequena, convertida para cinza e dividida em uma
    grade. Para cada bloco sao calculados media, desvio padrao, densidade de bordas, fluxo
    medio e fluxo radial medio. Isso cria uma entrada tabular simples o suficiente para MLP.

    Fontes:
    [OpenCV Canny] https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html
    [OpenCV Farneback Optical Flow] https://docs.opencv.org/4.x/d4/dee/tutorial_optical_flow.html
    [Artigo - Tarrio2015] https://doi.org/10.1109/iccv.2015.87
    '''

    imagem_pequena = cv2.resize(imagem_bgr, tamanho, interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(imagem_pequena, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(imagem_pequena, cv2.COLOR_BGR2HSV)
    edges = cv2.Canny(gray, 60, 160)

    if gray_anterior is None:
        fluxo_mag = np.zeros_like(gray, dtype=float)
        fluxo_radial = np.zeros_like(gray, dtype=float)
    else:
        flow = cv2.calcOpticalFlowFarneback(
            gray_anterior,
            gray,
            None,
            pyr_scale=0.5,
            levels=3,
            winsize=15,
            iterations=3,
            poly_n=5,
            poly_sigma=1.2,
            flags=0,
        )
        fluxo_mag = np.linalg.norm(flow, axis=2)
        yy, xx = np.mgrid[0:gray.shape[0], 0:gray.shape[1]]
        radial = np.stack([xx - gray.shape[1] * 0.5, yy - gray.shape[0] * 0.5], axis=2).astype(float)
        radial_norm = np.linalg.norm(radial, axis=2, keepdims=True) + 1e-6
        radial_unit = radial / radial_norm
        fluxo_radial = np.sum(flow * radial_unit, axis=2)

    features = {
        'gray_mean': float(gray.mean()),
        'gray_std': float(gray.std()),
        'edge_density': float((edges > 0).mean()),
        'sat_mean': float(hsv[:, :, 1].mean()),
        'sat_std': float(hsv[:, :, 1].std()),
        'flow_mag_mean': float(fluxo_mag.mean()),
        'flow_radial_mean': float(fluxo_radial.mean()),
    }

    hist = cv2.calcHist([gray], [0], None, [8], [0, 256]).flatten()
    hist = hist / max(float(hist.sum()), 1.0)
    for i, valor in enumerate(hist):
        features[f'gray_hist_{i}'] = float(valor)

    cols, rows = grid
    h, w = gray.shape
    for gy in range(rows):
        for gx in range(cols):
            y0, y1 = int(gy * h / rows), int((gy + 1) * h / rows)
            x0, x1 = int(gx * w / cols), int((gx + 1) * w / cols)
            prefix = f'g{gy}_{gx}'
            bloco_gray = gray[y0:y1, x0:x1]
            bloco_edges = edges[y0:y1, x0:x1]
            bloco_mag = fluxo_mag[y0:y1, x0:x1]
            bloco_radial = fluxo_radial[y0:y1, x0:x1]
            features[f'{prefix}_gray_mean'] = float(bloco_gray.mean())
            features[f'{prefix}_gray_std'] = float(bloco_gray.std())
            features[f'{prefix}_edge_density'] = float((bloco_edges > 0).mean())
            features[f'{prefix}_flow_mag'] = float(bloco_mag.mean())
            features[f'{prefix}_flow_radial'] = float(bloco_radial.mean())

    return features, gray


def extrair_features_estado_mlp(row):
    '''
    Extrai features escalares de atitude, IMU e compensacao visual.

    As features nao usam diretamente depth_min/depth_mean/depth_max para evitar vazamento do
    alvo. Elas usam apenas informacao que a camera monocular/estado do drone poderia ter no
    momento do frame.

    Fontes:
    [PX4 SensorCombined] https://docs.px4.io/main/en/msg_docs/SensorCombined.html
    [PX4 VehicleAttitude] https://docs.px4.io/main/en/msg_docs/VehicleAttitude
    '''

    colunas = [
        'roll', 'pitch', 'yaw',
        'gyro_x', 'gyro_y', 'gyro_z',
        'accel_x', 'accel_y', 'accel_z',
        'pan_comp_delta_rad',
    ]
    features = {}
    for coluna in colunas:
        valor = pd.to_numeric(row.get(coluna, 0.0), errors='coerce')
        features[coluna] = 0.0 if pd.isna(valor) else float(valor)

    features['gyro_norm'] = math.sqrt(features['gyro_x']**2 + features['gyro_y']**2 + features['gyro_z']**2)
    features['accel_norm'] = math.sqrt(features['accel_x']**2 + features['accel_y']**2 + features['accel_z']**2)
    features['tilt_abs'] = abs(features['roll']) + abs(features['pitch'])
    return features


def montar_dataset_mlp_depth(base_dir):
    '''
    Monta o dataset tabular usado pelo baseline MLP.

    A funcao percorre todas as runs de depth, extrai features de RGB, optical flow e IMU, e
    calcula os alvos compactos de profundidade. Cada amostra preserva run_id e sample_id para
    separacao temporal ou por run.

    Fontes:
    [Pandas DataFrame] https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html
    [OpenCV Optical Flow] https://docs.opencv.org/4.x/d4/dee/tutorial_optical_flow.html
    [scikit-learn supervised learning] https://scikit-learn.org/stable/supervised_learning.html
    '''

    linhas_x = []
    linhas_y = []
    linhas_meta = []
    metadados = listar_metadados_depth_mlp(base_dir)

    for metadata_path in metadados:
        run_dir = metadata_path.parent
        run_id = run_dir.name.replace('run_', '')
        df = pd.read_csv(metadata_path).sort_values('rgb_timestamp_s').reset_index(drop=True)
        gray_anterior = None

        for _, row in df.iterrows():
            try:
                imagem = carregar_rgb_mlp(row, run_dir)
                features_visuais, gray_atual = extrair_features_visuais_mlp(imagem, gray_anterior)
                features_estado = extrair_features_estado_mlp(row)
                alvos = calcular_alvos_depth_mlp(row, run_dir)
            except Exception as exc:
                print(f'Amostra ignorada em {run_id}: {exc}')
                continue

            gray_anterior = gray_atual
            linhas_x.append({**features_visuais, **features_estado})
            linhas_y.append(alvos)
            linhas_meta.append({
                'run_id': run_id,
                'sample_id': str(row.get('sample_id', '')).zfill(6),
                'rgb_timestamp_s': float(pd.to_numeric(row.get('rgb_timestamp_s', np.nan), errors='coerce')),
            })

    return pd.DataFrame(linhas_x), pd.DataFrame(linhas_y), pd.DataFrame(linhas_meta)


def separar_treino_validacao_teste_mlp(meta_df, random_state=42):
    '''
    Separa indices de treino, validacao e teste sem embaralhar frames de uma mesma run.

    Quando existem tres ou mais runs, a separacao e feita por run_id. Com uma unica run, a
    funcao usa uma divisao temporal 70/15/15, que e menos forte que validacao por run, mas
    evita misturar frames futuros no treino.

    Fontes:
    [scikit-learn model evaluation] https://scikit-learn.org/stable/model_selection.html
    [NumPy random generator] https://numpy.org/doc/stable/reference/random/generator.html
    '''

    n = len(meta_df)
    indices = np.arange(n)
    runs = meta_df['run_id'].dropna().unique()

    if len(runs) >= 3:
        rng = np.random.default_rng(random_state)
        runs = np.array(runs)
        rng.shuffle(runs)
        n_train = max(1, int(len(runs) * 0.7))
        n_val = max(1, int(len(runs) * 0.15))
        train_runs = set(runs[:n_train])
        val_runs = set(runs[n_train:n_train + n_val])
        test_runs = set(runs[n_train + n_val:])
        if not test_runs:
            test_runs = {runs[-1]}
            train_runs.discard(runs[-1])
        split = {
            'train': indices[meta_df['run_id'].isin(train_runs).to_numpy()],
            'val': indices[meta_df['run_id'].isin(val_runs).to_numpy()],
            'test': indices[meta_df['run_id'].isin(test_runs).to_numpy()],
            'modo': 'por run_id',
        }
    else:
        n_train = max(1, int(n * 0.70))
        n_val = max(1, int(n * 0.15))
        split = {
            'train': indices[:n_train],
            'val': indices[n_train:n_train + n_val],
            'test': indices[n_train + n_val:],
            'modo': 'temporal dentro da unica run disponivel',
        }

    if len(split['test']) == 0:
        split['test'] = split['val']
    if len(split['val']) == 0:
        split['val'] = split['test']
    return split


def avaliar_predicoes_mlp(y_true, y_pred, targets, nome_modelo, split_name):
    '''
    Calcula MAE, RMSE e correlacao por alvo para um conjunto de predicoes.

    A correlacao e informativa para saber se o modelo acompanha a tendencia dos frames, mesmo
    quando a escala absoluta ainda tem erro. MAE e RMSE indicam o erro direto das metricas de
    profundidade/risco.

    Fontes:
    [scikit-learn metrics] https://scikit-learn.org/stable/modules/model_evaluation.html
    [NumPy corrcoef] https://numpy.org/doc/stable/reference/generated/numpy.corrcoef.html
    '''

    linhas = []
    for idx, alvo in enumerate(targets):
        real = np.asarray(y_true[:, idx], dtype=float)
        pred = np.asarray(y_pred[:, idx], dtype=float)
        corr = np.nan
        if len(real) > 1 and np.std(real) > 1e-9 and np.std(pred) > 1e-9:
            corr = float(np.corrcoef(real, pred)[0, 1])
        linhas.append({
            'modelo': nome_modelo,
            'split': split_name,
            'alvo': alvo,
            'MAE': float(mean_absolute_error(real, pred)),
            'RMSE': float(mean_squared_error(real, pred) ** 0.5),
            'corr': corr,
        })
    return linhas


def treinar_avaliar_mlp_depth(X, y, meta_df, random_state=42):
    '''
    Treina a MLP simples e compara contra um baseline de media.

    A MLP recebe features normalizadas e alvos normalizados por StandardScaler. O baseline
    DummyRegressor prev? a media do treino, servindo como referencia minima: a MLP so e util
    se reduzir erro e/ou aumentar correlacao contra esse modelo burro.

    Fontes:
    [scikit-learn MLPRegressor] https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html
    [scikit-learn TransformedTargetRegressor] https://scikit-learn.org/stable/modules/generated/sklearn.compose.TransformedTargetRegressor.html
    [scikit-learn DummyRegressor] https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html
    '''

    targets = list(y.columns)
    split = separar_treino_validacao_teste_mlp(meta_df, random_state=random_state)
    X_values = X.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)
    y_values = y.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)

    train_idx = split['train']
    val_idx = split['val']
    test_idx = split['test']

    mlp = TransformedTargetRegressor(
        regressor=Pipeline([
            ('x_scaler', StandardScaler()),
            ('mlp', MLPRegressor(
                hidden_layer_sizes=(64, 32),
                activation='relu',
                solver='lbfgs',
                alpha=0.01,
                max_iter=2000,
                random_state=random_state,
            )),
        ]),
        transformer=StandardScaler(),
    )
    dummy = DummyRegressor(strategy='mean')

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        mlp.fit(X_values[train_idx], y_values[train_idx])
    dummy.fit(X_values[train_idx], y_values[train_idx])

    linhas = []
    predicoes = {}
    for split_name, idx in [('validacao', val_idx), ('teste', test_idx)]:
        pred_mlp = mlp.predict(X_values[idx])
        pred_dummy = dummy.predict(X_values[idx])
        predicoes[split_name] = {
            'idx': idx,
            'real': y_values[idx],
            'mlp': pred_mlp,
            'dummy': pred_dummy,
        }
        linhas.extend(avaliar_predicoes_mlp(y_values[idx], pred_mlp, targets, 'MLP', split_name))
        linhas.extend(avaliar_predicoes_mlp(y_values[idx], pred_dummy, targets, 'Media treino', split_name))

    return mlp, dummy, split, pd.DataFrame(linhas), predicoes


raiz_mlp = localizar_raiz_projeto_mlp()
X_mlp, y_mlp, meta_mlp = montar_dataset_mlp_depth(raiz_mlp)

if len(X_mlp) < 12:
    display(Markdown('Dataset insuficiente para treinar a MLP. Colete mais pares RGB/depth.'))
else:
    modelo_mlp_depth, baseline_media_depth, split_mlp, resultados_mlp_depth, predicoes_mlp_depth = treinar_avaliar_mlp_depth(
        X_mlp,
        y_mlp,
        meta_mlp,
    )

    display(Markdown(
        f'**Dataset MLP:** {len(X_mlp)} amostras, {X_mlp.shape[1]} features, '
        f'{y_mlp.shape[1]} alvos. Separacao usada: `{split_mlp["modo"]}`. '
        f'Treino={len(split_mlp["train"])}, validacao={len(split_mlp["val"])}, teste={len(split_mlp["test"])}.'
    ))

    display(resultados_mlp_depth.sort_values(['split', 'alvo', 'modelo']).reset_index(drop=True))

    resultados_teste = resultados_mlp_depth[resultados_mlp_depth['split'] == 'teste']
    fig_mae = go.Figure()
    for modelo in resultados_teste['modelo'].unique():
        parte = resultados_teste[resultados_teste['modelo'] == modelo]
        fig_mae.add_trace(go.Bar(x=parte['alvo'], y=parte['MAE'], name=modelo))
    fig_mae.update_layout(
        title='Erro MAE no teste: MLP x baseline de media',
        xaxis_title='Alvo supervisionado',
        yaxis_title='MAE',
        barmode='group',
        template='plotly_white',
    )
    fig_mae.show()

    targets = list(y_mlp.columns)
    pred_teste = predicoes_mlp_depth['teste']
    meta_teste = meta_mlp.iloc[pred_teste['idx']].reset_index(drop=True)
    alvo_plot = 'depth_close_5m_pct' if 'depth_close_5m_pct' in targets else targets[0]
    alvo_idx = targets.index(alvo_plot)

    fig_pred = go.Figure()
    fig_pred.add_trace(go.Scatter(
        x=meta_teste.index,
        y=pred_teste['real'][:, alvo_idx],
        mode='lines+markers',
        name='real',
    ))
    fig_pred.add_trace(go.Scatter(
        x=meta_teste.index,
        y=pred_teste['mlp'][:, alvo_idx],
        mode='lines+markers',
        name='MLP',
    ))
    fig_pred.add_trace(go.Scatter(
        x=meta_teste.index,
        y=pred_teste['dummy'][:, alvo_idx],
        mode='lines',
        name='media treino',
    ))
    fig_pred.update_layout(
        title=f'Predicao no teste para {alvo_plot}',
        xaxis_title='Amostras de teste em ordem temporal',
        yaxis_title=alvo_plot,
        template='plotly_white',
        hovermode='x unified',
    )
    fig_pred.show()

    tabela_predicoes = pd.DataFrame({
        'run_id': meta_teste['run_id'],
        'sample_id': meta_teste['sample_id'],
        f'{alvo_plot}_real': pred_teste['real'][:, alvo_idx],
        f'{alvo_plot}_mlp': pred_teste['mlp'][:, alvo_idx],
        f'{alvo_plot}_baseline_media': pred_teste['dummy'][:, alvo_idx],
    })
    display(Markdown('**Amostras de teste para inspecao:**'))
    display(tabela_predicoes.head(15))


**Dataset MLP:** 68 amostras, 88 features, 6 alvos. Separacao usada: `temporal dentro da unica run disponivel`. Treino=47, validacao=10, teste=11.

,modelo,split,alvo,MAE,RMSE,corr
0,MLP,teste,depth_close_10m_pct,4.764911,5.516976,-0.449209
1,Media treino,teste,depth_close_10m_pct,1.697505,2.176678,NaN
2,MLP,teste,depth_close_2m_pct,23.125753,23.989152,0.687189
3,Media treino,teste,depth_close_2m_pct,26.704656,28.360206,NaN
4,MLP,teste,depth_close_5m_pct,6.458410,8.167259,0.372330
5,Media treino,teste,depth_close_5m_pct,13.884411,15.085935,NaN
6,MLP,teste,depth_p10_m,0.681605,0.710368,0.822262
7,Media treino,teste,depth_p10_m,0.817645,0.896087,NaN
8,MLP,teste,depth_p50_m,2.141141,2.649365,-0.000804
9,Media treino,teste,depth_p50_m,2.700859,3.083470,NaN


**Amostras de teste para inspecao:**

,run_id,sample_id,depth_close_5m_pct_real,depth_close_5m_pct_mlp,depth_close_5m_pct_baseline_media
0,20260516_201713,000058,59.576959,59.994850,50.354761
1,20260516_201713,000059,56.283767,56.212314,50.354761
2,20260516_201713,000060,54.707217,57.196319,50.354761
3,20260516_201713,000061,57.505062,54.362260,50.354761
4,20260516_201713,000062,63.492621,55.822766,50.354761
5,20260516_201713,000063,72.373597,55.389953,50.354761
6,20260516_201713,000064,67.714459,56.551755,50.354761
7,20260516_201713,000065,67.686610,55.517066,50.354761
8,20260516_201713,000066,68.757717,63.381243,50.354761
9,20260516_201713,000067,69.337544,62.607757,50.354761
